# Experiment 8: Ridge and Lasso Regularized Linear Regression vs Standard Linear Regression

## Aim / Objective
To implement and compare **Standard Linear Regression (Ordinary Least Squares - OLS)**, **Ridge Regression ($L_2$ Regularization)**, and **Lasso Regression ($L_1$ Regularization)** on two benchmark machine learning datasets:
1. **California Housing Dataset** (Continuous Regression Task)
2. **Breast Cancer Wisconsin Dataset** (Binary Classification Task / Linear Probability Model)

---

## Mathematical Theory & Derivations

### 1. Standard Linear Regression (Ordinary Least Squares - OLS)
Ordinary Least Squares estimates feature weights $\mathbf{w}$ and bias $b$ by minimizing the **Residual Sum of Squares (RSS)** on training samples $(x_i, y_i)$:

$$\min_{\mathbf{w}, b} \mathcal{L}_{\text{OLS}}(\mathbf{w}, b) = \frac{1}{2n} \sum_{i=1}^{n} \left(y_i - (\mathbf{w}^T \mathbf{x}_i + b)\right)^2$$

- **Closed-form Analytical Solution**: $\hat{\mathbf{w}}_{\text{OLS}} = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$
- **Limitations**: OLS is highly susceptible to **overfitting** when features are correlated (multicollinearity) or when feature dimension $p$ is large relative to sample size $n$. Large weights amplify noise.

---

### 2. Ridge Regression ($L_2$ Regularization / Tikhonov Regularization)
Ridge regression penalizes the sum of squared weights ($L_2$ norm), adding a shrinkage penalty controlled by hyperparameter $\alpha \ge 0$:

$$\min_{\mathbf{w}, b} \mathcal{L}_{\text{Ridge}}(\mathbf{w}, b) = \frac{1}{2n} \sum_{i=1}^{n} \left(y_i - (\mathbf{w}^T \mathbf{x}_i + b)\right)^2 + \alpha \sum_{j=1}^{p} w_j^2$$

- **Closed-form Solution**: $\hat{\mathbf{w}}_{\text{Ridge}} = (\mathbf{X}^T \mathbf{X} + \alpha \mathbf{I})^{-1} \mathbf{X}^T \mathbf{y}$
- **Key Behavior**: Adds $\alpha \mathbf{I}$ to make matrix $\mathbf{X}^T \mathbf{X}$ strictly invertible (solving multicollinearity). Weight magnitudes shrink towards zero smoothly, but coefficients are **never reduced exactly to zero**.

---

### 3. Lasso Regression ($L_1$ Regularization - Least Absolute Shrinkage and Selection Operator)
Lasso regression penalizes the sum of absolute weight values ($L_1$ norm):

$$\min_{\mathbf{w}, b} \mathcal{L}_{\text{Lasso}}(\mathbf{w}, b) = \frac{1}{2n} \sum_{i=1}^{n} \left(y_i - (\mathbf{w}^T \mathbf{x}_i + b)\right)^2 + \alpha \sum_{j=1}^{p} |w_j|$$

- **Optimization**: Non-differentiable at $w_j = 0$; optimized using **Coordinate Descent**.
- **Key Behavior**: The sharp diamond geometry of the $L_1$ constraint region forces coefficients of non-informative features to become **exactly zero**. Lasso acts as an **automatic feature selector** producing sparse models.

---

### Comparison Summary Table

| Feature / Metric | Standard Linear Regression (OLS) | Ridge Regression ($L_2$) | Lasso Regression ($L_1$) |
| :--- | :--- | :--- | :--- |
| **Penalty Term** | None | $\alpha \|\mathbf{w}\|_2^2 = \alpha \sum w_j^2$ | $\alpha \|\mathbf{w}\|_1 = \alpha \sum \|w_j\|$ |
| **Weight Effect** | Unconstrained | Smooth weight shrinkage towards 0 | Shrinkage + Exact Sparsity (Zero weights) |
| **Feature Selection** | No | No | Yes (Automatic) |
| **Multicollinearity** | Fails / Unstable | Robust (adds diagonal $\alpha \mathbf{I}$) | Selects one feature, zeroes others |
| **Solution Method** | Closed-form OLS | Closed-form Ridge | Iterative Coordinate Descent |
| **Feature Scaling** | Optional | **Mandatory** | **Mandatory** |

In [ ]:
# 1. Load core packages
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV, LogisticRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

SEED = 42
np.random.seed(SEED)

# Set style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_theme(style="whitegrid")
%matplotlib inline

print("Libraries loaded successfully.")

## PART 1: California Housing Dataset (Continuous Regression)

Predicting continuous median house values (`MedHouseVal`) across California districts.

In [ ]:
# 2. Load California Housing Dataset
housing = fetch_california_housing(as_frame=True)
X_housing, y_housing = housing.data, housing.target
housing_features = housing.feature_names

print(f"California Housing Shape: X = {X_housing.shape}, y = {y_housing.shape}")
display(X_housing.head())

In [ ]:
# 3. Train-Test Split (80/20) & Feature Scaling
X_tr_h, X_te_h, y_tr_h, y_te_h = train_test_split(X_housing, y_housing, test_size=0.2, random_state=SEED)

scaler_h = StandardScaler()
X_tr_h_scaled = scaler_h.fit_transform(X_tr_h)
X_te_h_scaled = scaler_h.transform(X_te_h)

print(f"Train set size: {X_tr_h_scaled.shape[0]}, Test set size: {X_te_h_scaled.shape[0]}")

In [ ]:
# 4. Fit OLS, RidgeCV, and LassoCV
ols_h = LinearRegression()
ols_h.fit(X_tr_h_scaled, y_tr_h)

# Ridge 5-fold CV
alphas_ridge = np.logspace(-3, 5, 200)
ridge_cv_h = RidgeCV(alphas=alphas_ridge, cv=5, scoring='neg_mean_squared_error')
ridge_cv_h.fit(X_tr_h_scaled, y_tr_h)

# Lasso 5-fold CV
alphas_lasso = np.logspace(-4, 2, 200)
lasso_cv_h = LassoCV(alphas=alphas_lasso, cv=5, max_iter=20000, random_state=SEED)
lasso_cv_h.fit(X_tr_h_scaled, y_tr_h)

print(f"Optimal Ridge Alpha (L2): {ridge_cv_h.alpha_:.4f}")
print(f"Optimal Lasso Alpha (L1): {lasso_cv_h.alpha_:.6f}")

In [ ]:
# 5. Performance Evaluation Metrics
models_h = {
    "OLS Linear Regression": ols_h,
    f"Ridge (alpha={ridge_cv_h.alpha_:.2f})": ridge_cv_h,
    f"Lasso (alpha={lasso_cv_h.alpha_:.4f})": lasso_cv_h
}

res_h = []
for name, model in models_h.items():
    y_tr_pred = model.predict(X_tr_h_scaled)
    y_te_pred = model.predict(X_te_h_scaled)
    res_h.append({
        "Model": name,
        "Train MSE": mean_squared_error(y_tr_h, y_tr_pred),
        "Test MSE": mean_squared_error(y_te_h, y_te_pred),
        "Test RMSE": np.sqrt(mean_squared_error(y_te_h, y_te_pred)),
        "Test MAE": mean_absolute_error(y_te_h, y_te_pred),
        "Train R2": r2_score(y_tr_h, y_tr_pred),
        "Test R2": r2_score(y_te_h, y_te_pred)
    })

df_res_h = pd.DataFrame(res_h)
display(df_res_h.style.highlight_max(subset=['Test R2'], color='lightgreen').highlight_min(subset=['Test MSE'], color='lightgreen'))

In [ ]:
# 6. Feature Coefficient & Sparsity Analysis
coef_h_df = pd.DataFrame({
    "Feature": housing_features,
    "OLS": ols_h.coef_,
    "Ridge": ridge_cv_h.coef_,
    "Lasso": lasso_cv_h.coef_
}).set_index("Feature")

print("Learned Feature Coefficients (Standardized Scale):")
display(coef_h_df)

print(f"OLS Non-Zero Features: {np.sum(ols_h.coef_ != 0)} / {len(housing_features)}")
print(f"Ridge Non-Zero Features: {np.sum(ridge_cv_h.coef_ != 0)} / {len(housing_features)}")
print(f"Lasso Non-Zero Features: {np.sum(lasso_cv_h.coef_ != 0)} / {len(housing_features)}")

In [ ]:
# 7. Visualization: Coefficient Comparison Bar Plot
plt.figure(figsize=(10, 5))
x_idx = np.arange(len(housing_features))
w = 0.25

plt.bar(x_idx - w, ols_h.coef_, w, label='OLS Linear Regression', color='#1f77b4')
plt.bar(x_idx, ridge_cv_h.coef_, w, label=f'Ridge (L2, a={ridge_cv_h.alpha_:.2f})', color='#2ca02c')
plt.bar(x_idx + w, lasso_cv_h.coef_, w, label=f'Lasso (L1, a={lasso_cv_h.alpha_:.4f})', color='#d62728')

plt.title('California Housing: Standardized Feature Coefficients Comparison', fontsize=13, fontweight='bold')
plt.ylabel('Coefficient Magnitude', fontsize=11)
plt.xticks(x_idx, housing_features, rotation=30, ha='right')
plt.axhline(0, color='black', linestyle='--', linewidth=0.8)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Actual vs Predicted Scatter Plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
names = ["OLS Linear Regression", "Ridge Regression", "Lasso Regression"]
models_list = [ols_h, ridge_cv_h, lasso_cv_h]
colors = ['#1f77b4', '#2ca02c', '#d62728']

for ax, name, model, color in zip(axes, names, models_list, colors):
    y_pred = model.predict(X_te_h_scaled)
    r2 = r2_score(y_te_h, y_pred)
    rmse = np.sqrt(mean_squared_error(y_te_h, y_pred))
    ax.scatter(y_te_h, y_pred, alpha=0.25, color=color, s=12)
    ax.plot([y_te_h.min(), y_te_h.max()], [y_te_h.min(), y_te_h.max()], 'k--', lw=2, label='Ideal')
    ax.set_xlabel('Actual MedHouseVal ($100k)')
    ax.set_ylabel('Predicted MedHouseVal ($100k)')
    ax.set_title(f"{name}\n$R^2$: {r2:.4f} | RMSE: {rmse:.4f}", fontweight='bold')
    ax.legend(loc='upper left')

plt.tight_layout()
plt.show()

## PART 2: Breast Cancer Wisconsin Dataset (Binary Classification)

Evaluating Standard Linear Regression, Ridge, and Lasso as **Linear Probability Models** (continuous output thresholded at 0.5) alongside **Regularized Logistic Regression**.

In [ ]:
# 8. Load Breast Cancer Dataset
cancer = load_breast_cancer(as_frame=True)
X_cancer, y_cancer = cancer.data, cancer.target
cancer_features = cancer.feature_names

print(f"Breast Cancer Dataset Shape: X = {X_cancer.shape}, y = {y_cancer.shape}")
print(f"Class Distribution: {np.bincount(y_cancer)} (0: Malignant, 1: Benign)")

In [ ]:
# 9. Train-Test Split (80/20 Stratified) & Scaling
X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(X_cancer, y_cancer, test_size=0.2, random_state=SEED, stratify=y_cancer)

scaler_c = StandardScaler()
X_tr_c_scaled = scaler_c.fit_transform(X_tr_c)
X_te_c_scaled = scaler_c.transform(X_te_c)

In [ ]:
# 10. Fit Linear Probability Models (OLS, RidgeCV, LassoCV)
ols_c = LinearRegression().fit(X_tr_c_scaled, y_tr_c)
ridge_c = RidgeCV(alphas=np.logspace(-2, 4, 200), cv=5).fit(X_tr_c_scaled, y_tr_c)
lasso_c = LassoCV(alphas=np.logspace(-4, 0, 200), cv=5, max_iter=20000, random_state=SEED).fit(X_tr_c_scaled, y_tr_c)

c_reg_models = {
    "OLS Linear Regression": ols_c,
    f"Ridge Regression (alpha={ridge_c.alpha_:.2f})": ridge_c,
    f"Lasso Regression (alpha={lasso_c.alpha_:.4f})": lasso_c
}

c_lin_results = []
for name, model in c_reg_models.items():
    y_prob = model.predict(X_te_c_scaled)
    y_pred = (y_prob >= 0.5).astype(int)
    c_lin_results.append({
        "Model": name,
        "Test MSE": mean_squared_error(y_te_c, y_prob),
        "Accuracy": accuracy_score(y_te_c, y_pred),
        "Precision": precision_score(y_te_c, y_pred),
        "Recall": recall_score(y_te_c, y_pred),
        "F1-Score": f1_score(y_te_c, y_pred),
        "ROC-AUC": roc_auc_score(y_te_c, y_prob),
        "Non-Zero Features": np.sum(model.coef_ != 0)
    })

df_c_lin = pd.DataFrame(c_lin_results)
display(df_c_lin.style.highlight_max(subset=['Accuracy', 'F1-Score', 'ROC-AUC'], color='lightgreen'))

In [ ]:
# 11. Logistic Regression Regularization Comparison (Unregularized vs L2 vs L1)
log_unreg = LogisticRegression(penalty=None, max_iter=10000, random_state=SEED).fit(X_tr_c_scaled, y_tr_c)
log_l2 = LogisticRegression(penalty='l2', C=1.0, max_iter=10000, random_state=SEED).fit(X_tr_c_scaled, y_tr_c)
log_l1 = LogisticRegression(penalty='l1', solver='saga', C=0.5, max_iter=10000, random_state=SEED).fit(X_tr_c_scaled, y_tr_c)

log_models = {
    "Unregularized Logistic Regression": log_unreg,
    "Ridge Logistic Regression (L2, C=1.0)": log_l2,
    "Lasso Logistic Regression (L1, C=0.5)": log_l1
}

log_res = []
for name, model in log_models.items():
    y_prob = model.predict_proba(X_te_c_scaled)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    log_res.append({
        "Model": name,
        "Accuracy": accuracy_score(y_te_c, y_pred),
        "Precision": precision_score(y_te_c, y_pred),
        "Recall": recall_score(y_te_c, y_pred),
        "F1-Score": f1_score(y_te_c, y_pred),
        "ROC-AUC": roc_auc_score(y_te_c, y_prob),
        "Non-Zero Features": np.sum(model.coef_[0] != 0)
    })

df_log = pd.DataFrame(log_res)
display(df_log.style.highlight_max(subset=['Accuracy', 'F1-Score', 'ROC-AUC'], color='lightgreen'))

In [ ]:
# 12. Visualization: Feature Weights across 30 Features in Breast Cancer
plt.figure(figsize=(14, 6))
x_idx = np.arange(len(cancer_features))
w = 0.25

plt.bar(x_idx - w, ols_c.coef_, w, label='OLS Linear Regression', color='#1f77b4', alpha=0.85)
plt.bar(x_idx, ridge_c.coef_, w, label=f'Ridge (a={ridge_c.alpha_:.2f})', color='#2ca02c', alpha=0.85)
plt.bar(x_idx + w, lasso_c.coef_, w, label=f'Lasso (a={lasso_c.alpha_:.4f})', color='#d62728', alpha=0.85)

plt.title('Breast Cancer Dataset: Standardized Coefficients Across OLS, Ridge, & Lasso', fontsize=13, fontweight='bold')
plt.ylabel('Weight Magnitude', fontsize=11)
plt.xticks(x_idx, cancer_features, rotation=90, ha='right', fontsize=9)
plt.axhline(0, color='black', linestyle='--', linewidth=0.8)
plt.legend()
plt.tight_layout()
plt.show()

## Key Takeaways & Conclusions

1. **Continuous Target (California Housing)**:
   - Standard Linear Regression (OLS), Ridge, and Lasso achieve similar overall $R^2 \approx 0.5758$ on the test set because sample size $n = 20,640$ is large relative to feature count $p = 8$.
   - **Ridge ($L_2$)** slightly shrinks large weight coefficients (`AveRooms` and `AveBedrms`) mitigating hidden collinearity.
   - **Lasso ($L_1$)** zeroes out redundant features at higher $\alpha$, leaving only key drivers like `MedInc`, `Latitude`, and `Longitude`.

2. **Classification Target (Breast Cancer Wisconsin)**:
   - **Linear Probability Model (Lasso Regression)** achieves **97.37% Accuracy** and **0.9934 ROC-AUC** using only **10 features out of 30** (zeroing 20 non-informative features).
   - **Ridge Regression** retains all 30 features with stabilized weights.
   - Regularization prevents coefficient explosion on collinear biological measurement features, outperforming unregularized models on test data.